# Lab | Reinforcement Learning

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

## Task 1: Environment Exploration

In [ ]:
# 1.1, 1.2
env_fl = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="ansi")
print("FrozenLake Observation Space:", env_fl.observation_space)
print("FrozenLake Action Space:", env_fl.action_space)
print("Action Meanings: 0: Left, 1: Down, 2: Right, 3: Up")

# 1.3
print("\nRunning 5 random episodes for FrozenLake:")
for i in range(5):
    state, _ = env_fl.reset()
    done = False
    total_reward = 0
    steps = 0
    while not done:
        action = env_fl.action_space.sample()
        state, reward, terminated, truncated, _ = env_fl.step(action)
        total_reward += reward
        steps += 1
        done = terminated or truncated
    print(f"Episode {i+1}: Reward = {total_reward}, Steps = {steps}")

# 1.4
print("\nFrozenLake Grid:")
print(env_fl.render())

In [ ]:
# 1.5
env_taxi = gym.make("Taxi-v3", render_mode="ansi")
print("Taxi Observation Space:", env_taxi.observation_space)
print("Taxi Action Space:", env_taxi.action_space)

print("\nRunning 5 random episodes for Taxi:")
for i in range(5):
    state, _ = env_taxi.reset()
    done = False
    total_reward = 0
    steps = 0
    while not done:
        action = env_taxi.action_space.sample()
        state, reward, terminated, truncated, _ = env_taxi.step(action)
        total_reward += reward
        steps += 1
        done = terminated or truncated
    print(f"Episode {i+1}: Reward = {total_reward}, Steps = {steps}")

### 1.6
The FrozenLake state space is discrete with 16 states (4x4 grid). The Taxi state space is much larger with 500 discrete states (representing taxi location, passenger location, and destination). Both have a discrete action space, but Taxi has 6 actions (Move South, North, East, West, Pick up, Drop off) while FrozenLake has 4 (Left, Down, Right, Up). Taxi is harder because of the significantly larger state space and the requirement to coordinate pick-up and drop-off actions correctly, whereas FrozenLake is a simpler navigation task.

## Task 2: Q-Learning on FrozenLake

In [ ]:
# 2.1, 2.2, 2.3, 2.4
n_states = env_fl.observation_space.n
n_actions = env_fl.action_space.n
Q_fl = np.zeros((n_states, n_actions))

alpha = 0.8
gamma = 0.95
epsilon = 1.0
epsilon_decay = 0.995
min_epsilon = 0.01
episodes = 10000

rewards_fl = []

for episode in range(episodes):
    state, _ = env_fl.reset()
    done = False
    total_reward = 0
    
    while not done:
        if np.random.uniform(0, 1) < epsilon:
            action = env_fl.action_space.sample()
        else:
            action = np.argmax(Q_fl[state, :])
            
        next_state, reward, terminated, truncated, _ = env_fl.step(action)
        
        Q_fl[state, action] = Q_fl[state, action] + alpha * (reward + gamma * np.max(Q_fl[next_state, :]) - Q_fl[state, action])
        
        state = next_state
        total_reward += reward
        done = terminated or truncated
        
    epsilon = max(min_epsilon, epsilon * epsilon_decay)
    rewards_fl.append(total_reward)

# 2.5
plt.figure(figsize=(10, 5))
plt.plot(pd.Series(rewards_fl).rolling(window=100).mean())
plt.title("FrozenLake: Cumulative Reward (Rolling Average)")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.show()

In [ ]:
# 2.6
print("Final Q-table for FrozenLake:")
print(Q_fl)
print("\nAction with highest Q-value for start state (0):", np.argmax(Q_fl[0, :]))

### 2.6
The agent learned a policy that successfully navigates to the goal. For the start state, the agent likely prefers moving Down or Right (depending on ties), which are the directions towards the goal. The learned policy makes intuitive sense as it avoids holes (where Q-values remain low) and prioritizes paths leading to the goal tile (reward 1).

## Task 3: Q-Learning on Taxi

In [ ]:
# 3.1, 3.2, 3.3, 3.4
n_states_taxi = env_taxi.observation_space.n
n_actions_taxi = env_taxi.action_space.n
Q_taxi = np.zeros((n_states_taxi, n_actions_taxi))

epsilon = 1.0
rewards_taxi = []
episodes_taxi = 20000

for episode in range(episodes_taxi):
    state, _ = env_taxi.reset()
    done = False
    total_reward = 0
    
    while not done:
        if np.random.uniform(0, 1) < epsilon:
            action = env_taxi.action_space.sample()
        else:
            action = np.argmax(Q_taxi[state, :])
            
        next_state, reward, terminated, truncated, _ = env_taxi.step(action)
        
        Q_taxi[state, action] = Q_taxi[state, action] + alpha * (reward + gamma * np.max(Q_taxi[next_state, :]) - Q_taxi[state, action])
        
        state = next_state
        total_reward += reward
        done = terminated or truncated
        
    epsilon = max(min_epsilon, epsilon * epsilon_decay)
    rewards_taxi.append(total_reward)

plt.figure(figsize=(10, 5))
plt.plot(pd.Series(rewards_taxi).rolling(window=100).mean())
plt.title("Taxi: Average Reward per 100 Episodes")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.show()

In [ ]:
# 3.5
test_episodes = 100
test_rewards = []
successes = 0

for _ in range(test_episodes):
    state, _ = env_taxi.reset()
    done = False
    total_reward = 0
    while not done:
        action = np.argmax(Q_taxi[state, :])
        state, reward, terminated, truncated, _ = env_taxi.step(action)
        total_reward += reward
        done = terminated or truncated
    test_rewards.append(total_reward)
    if total_reward > 0:
        successes += 1

print(f"Average Test Reward: {np.mean(test_rewards)}")
print(f"Success Rate: {successes/test_episodes * 100}%")

### 3.6
The Taxi environment training curve starts much lower than FrozenLake due to the negative rewards for every step and large penalties for wrong pick-ups/drop-offs. It takes significantly more episodes (usually around 5,000-10,000) for the reward to stabilize and become positive, as the agent must first discover the correct sequence of pick-up and drop-off actions among the 500 states.

## Task 4: SARSA Comparison

In [ ]:
# 4.1, 4.2
Q_sarsa = np.zeros((n_states_taxi, n_actions_taxi))
epsilon = 1.0
rewards_sarsa = []

for episode in range(episodes_taxi):
    state, _ = env_taxi.reset()
    
    if np.random.uniform(0, 1) < epsilon:
        action = env_taxi.action_space.sample()
    else:
        action = np.argmax(Q_sarsa[state, :])
        
    done = False
    total_reward = 0
    
    while not done:
        next_state, reward, terminated, truncated, _ = env_taxi.step(action)
        
        if np.random.uniform(0, 1) < epsilon:
            next_action = env_taxi.action_space.sample()
        else:
            next_action = np.argmax(Q_sarsa[next_state, :])
            
        Q_sarsa[state, action] = Q_sarsa[state, action] + alpha * (reward + gamma * Q_sarsa[next_state, next_action] - Q_sarsa[state, action])
        
        state = next_state
        action = next_action
        total_reward += reward
        done = terminated or truncated
        
    epsilon = max(min_epsilon, epsilon * epsilon_decay)
    rewards_sarsa.append(total_reward)

In [ ]:
# 4.3
plt.figure(figsize=(10, 5))
plt.plot(pd.Series(rewards_taxi).rolling(window=100).mean(), label="Q-Learning")
plt.plot(pd.Series(rewards_sarsa).rolling(window=100).mean(), label="SARSA")
plt.title("Q-Learning vs SARSA on Taxi-v3")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.legend()
plt.show()

In [ ]:
# 4.4
sarsa_test_rewards = []
sarsa_successes = 0

for _ in range(test_episodes):
    state, _ = env_taxi.reset()
    done = False
    total_reward = 0
    while not done:
        action = np.argmax(Q_sarsa[state, :])
        state, reward, terminated, truncated, _ = env_taxi.step(action)
        total_reward += reward
        done = terminated or truncated
    sarsa_test_rewards.append(total_reward)
    if total_reward > 0:
        sarsa_successes += 1

print(f"SARSA Average Test Reward: {np.mean(sarsa_test_rewards)}")
print(f"SARSA Success Rate: {sarsa_successes/test_episodes * 100}%")

### 4.5
- **Convergence:** Q-Learning often converges faster to the optimal policy because it is off-policy and directly learns the greedy policy. SARSA, being on-policy, can be more conservative during training as it accounts for the exploratory actions it might take.
- **Final Reward:** In this deterministic environment (Taxi-v3), both typically reach similar final performance, though Q-Learning might find a slightly more aggressive optimal path.
- **Fundamental Difference:** Q-Learning is **off-policy**: it updates its Q-values assuming the best possible action will be taken in the next state, regardless of the actual action taken. SARSA is **on-policy**: it updates its Q-values based on the actual next action chosen by the current policy (including exploration).
- **Situations:** SARSA is preferred when the cost of exploration is high or dangerous (e.g., robotics), as it learns a safer policy that accounts for potential mistakes. Q-Learning is preferred when we want to learn the absolute optimal strategy and can afford some "risky" exploration during training.